# `ann_df` Isotope Peak Labelling

This notebook traces exactly how isotope labels end up in `ann_df` and
analyses their distribution in the NIST20 processed data.

## Pipeline summary

```
NIST20 raw peak string
  └─ parse_product_isotope()   # detects trailing 'i' suffix → integer offset
       └─ parse_annotations()  # builds ann_isotopes list per spectrum
            └─ 02_prepare_proc.py  → ann_df.pkl  (column: ann_isotopes)
                 └─ fit_ann_library()  # filters isotopes > max_isotope
```

### Annotation string format (NIST20)

Each peak row in NIST20 looks like:
```
<mz> <int> <annotation_string>
```
e.g.:
```
100.112  8521  p-C4H8
101.120  1043  p-C4H8+1i
```

A trailing `+ki` (or `-ki`) in the annotation marks the peak as the
**M+k** (or M-k) isotopologue of the fragment ion.  Bare `i` → k=1.

### `parse_product_isotope` logic (data_utils.py:976)

```python
# chunk ends in 'i' (but not 'Si') → isotope token
if chunk[-1] == 'i' and chunk[-2:] != 'Si':
    chunk = chunk[:-1]          # strip 'i'
    isotope = int(chunk) if chunk else 1  # bare 'i' → 1
    if prefix == '-': isotope = -isotope
```

Result: `isotope` is an integer ≥ 0 for all normal annotations.
Negative values are technically possible but treated as invalid downstream.


In [ ]:
import os
import re
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── paths ──────────────────────────────────────────────────────────────────
PROC_DP = "../../data/proc/nist20"
ANN_DF_FP = os.path.join(PROC_DP, "ann_df.pkl")
SPEC_DF_FP = os.path.join(PROC_DP, "spec_df.pkl")

## 1  Parse-time demo: `parse_product_isotope`

Walk through representative annotation strings to show what `isotope` value
each produces.

In [ ]:
import importlib
import sys

sys.path.insert(0, "../../src")
from fragnnet.utils.data_utils import parse_product_isotope
from pyteomics.mass import Composition

# Representative examples from NIST20 annotation strings
# precursor = [M+H]+ of C6H12O  → C6H13O
precursor = "C6H13O"

examples = [
    ("p-H2O",        "monoisotopic product ion (M+0)"),
    ("p-H2O+1i",     "M+1 isotopologue of p-H2O"),
    ("p-H2O+2i",     "M+2 isotopologue of p-H2O"),
    ("p-H2O+i",      "same as +1i (bare 'i')"),
    ("p-CH2O",       "different neutral loss, monoisotopic"),
    ("p-CH2O+1i",    "M+1 of different loss"),
    ("p",            "intact precursor (monoisotopic)"),
    ("p+1i",         "M+1 of intact precursor"),
]

rows = []
for annot, desc in examples:
    product, adducts, losses, isotope, charge = parse_product_isotope(annot, precursor)
    rows.append({"annotation": annot, "isotope": isotope, "description": desc})

pd.DataFrame(rows)

## 2  Load `ann_df`

In [ ]:
ann_df = pd.read_pickle(ANN_DF_FP)
print("ann_df shape:", ann_df.shape)
print("columns:", ann_df.columns.tolist())
ann_df.head(3)

## 3  Explode to peak level

`ann_isotopes` is a list-per-spectrum in `ann_df`.  Explode to get one row
per annotated peak.

In [ ]:
peaks = ann_df[["spec_id", "ann_peak_mzs", "ann_products", "ann_losses",
                "ann_isotopes", "ann_exact_mzs"]].copy()

peaks["joint"] = peaks.apply(
    lambda r: list(zip(r["ann_peak_mzs"], r["ann_products"],
                       r["ann_losses"], r["ann_isotopes"], r["ann_exact_mzs"])),
    axis=1,
)
peaks = peaks[["spec_id", "joint"]].explode("joint")
peaks[["peak_mz", "product", "loss", "isotope", "exact_mz"]] = pd.DataFrame(
    peaks["joint"].tolist(), index=peaks.index
)
peaks = peaks.drop(columns=["joint"])
peaks["isotope"] = peaks["isotope"].astype(int)
peaks["peak_mz"] = peaks["peak_mz"].astype(float)
peaks["exact_mz"] = peaks["exact_mz"].astype(float)

print(f"Total annotated peaks: {len(peaks):,}")
peaks.head(5)

## 4  Isotope label distribution

In [ ]:
iso_counts = peaks["isotope"].value_counts().sort_index()
iso_frac = iso_counts / iso_counts.sum()

df_iso = pd.DataFrame({"count": iso_counts, "fraction": iso_frac})
print(df_iso.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
iso_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Isotope label (0 = monoisotopic, 1 = M+1, …)")
ax.set_ylabel("Number of annotated peaks")
ax.set_title("Isotope label distribution in ann_df (NIST20)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(
        f"{p.get_height() / iso_counts.sum():.1%}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center", va="bottom", fontsize=9,
    )
plt.tight_layout()
plt.show()

## 5  Spectra with ≥1 isotope peak

In [ ]:
n_total_spectra = ann_df["spec_id"].nunique()
has_iso = peaks[peaks["isotope"] > 0]["spec_id"].nunique()
print(f"Spectra with ≥1 M+k peak   : {has_iso:,} / {n_total_spectra:,}  "
      f"({has_iso/n_total_spectra:.1%})")

# Distribution of isotope-peak fraction per spectrum
per_spec = peaks.groupby("spec_id")["isotope"].agg(
    total="count",
    n_iso=lambda x: (x > 0).sum(),
).reset_index()
per_spec["iso_frac"] = per_spec["n_iso"] / per_spec["total"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(per_spec["iso_frac"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Fraction of annotated peaks that are isotopes")
axes[0].set_ylabel("Number of spectra")
axes[0].set_title("Per-spectrum isotope peak fraction")

axes[1].hist(per_spec["n_iso"], bins=range(0, 20), color="darkorange", edgecolor="white")
axes[1].set_xlabel("Number of isotope peaks per spectrum")
axes[1].set_ylabel("Number of spectra")
axes[1].set_title("Per-spectrum isotope peak count")

plt.tight_layout()
plt.show()

print(per_spec["iso_frac"].describe())

## 6  m/z error for isotope vs monoisotopic peaks

`exact_mz` is computed as `isotope * NEUTRON_MASS + monoisotopic_mass`.
Check how well observed `peak_mz` agrees.

In [ ]:
peaks["mz_err"] = (peaks["peak_mz"] - peaks["exact_mz"]).abs()
peaks["mz_err_ppm"] = peaks["mz_err"] / peaks["peak_mz"].clip(lower=50) * 1e6

err_stats = peaks.groupby("isotope")[["mz_err", "mz_err_ppm"]].describe(percentiles=[0.5, 0.9, 0.99])
print(err_stats[["mz_err", "mz_err_ppm"]].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for iso_val, grp in peaks[peaks["isotope"].isin([0, 1, 2])].groupby("isotope"):
    ax.hist(
        grp["mz_err"].clip(upper=0.05),
        bins=60,
        alpha=0.6,
        label=f"M+{iso_val}",
        density=True,
    )
ax.set_xlabel("|peak_mz − exact_mz| Da  (clipped at 0.05)")
ax.set_ylabel("Density")
ax.set_title("m/z error by isotope label")
ax.legend()
plt.tight_layout()
plt.show()

## 7  Most common isotope-labelled product formulas

In [ ]:
iso_peaks = peaks[peaks["isotope"] > 0].copy()

print(f"M+1 peaks — top 20 product formulas:")
display(
    iso_peaks[iso_peaks["isotope"] == 1]["product"]
    .value_counts()
    .head(20)
    .reset_index()
    .rename(columns={"product": "formula", "count": "n_peaks"})
)

print(f"\nM+2 peaks — top 20 product formulas:")
display(
    iso_peaks[iso_peaks["isotope"] == 2]["product"]
    .value_counts()
    .head(20)
    .reset_index()
    .rename(columns={"product": "formula", "count": "n_peaks"})
)

## 8  Example spectra: raw NIST annotation string vs parsed labels

Show the raw annotation string side-by-side with the parsed `(product, isotope)`
for a handful of spectra that contain isotope peaks.

In [ ]:
spec_df = pd.read_pickle(SPEC_DF_FP)

# pick spectra that have at least 2 isotope peaks
rich_iso_ids = per_spec[per_spec["n_iso"] >= 2]["spec_id"].values[:5]

for sid in rich_iso_ids:
    print(f"\n{'='*70}")
    print(f"spec_id = {sid}")
    row = spec_df[spec_df["spec_id"] == sid].iloc[0]
    print(f"  prec_mz={row.get('prec_mz', '?'):.4f}  "
          f"inst={row.get('inst_type','?')}  frag={row.get('frag_mode','?')}")
    sub = peaks[peaks["spec_id"] == sid].sort_values("peak_mz")
    for _, r in sub.iterrows():
        flag = "  ◄ isotope" if r["isotope"] > 0 else ""
        print(f"  mz={r['peak_mz']:8.4f}  product={r['product']:20s}  "
              f"iso={r['isotope']}  err={r['mz_err']:.4f} Da{flag}")

## 9  `fit_ann_library` filtering: what gets dropped

`fit_ann_library` (graff/data_utils.py) drops annotations where
`isotopes > max_isotope` (default 2) or `isotopes < 0`.  Check how many
peaks would be affected.

In [ ]:
for max_iso in [1, 2, 3]:
    invalid = (peaks["isotope"] > max_iso) | (peaks["isotope"] < 0)
    n_drop = invalid.sum()
    print(f"max_isotope={max_iso}: {n_drop:,} peaks dropped "
          f"({n_drop/len(peaks):.3%} of all annotated peaks)")

## 10  Isotope intensity share per spectrum

Join with actual peak intensities to see what fraction of annotated
intensity is from isotope peaks.

In [ ]:
# Build a mz→intensity map from spec_df
spec_peaks = spec_df[["spec_id", "peaks"]].copy()
spec_peaks = spec_peaks.explode("peaks")
spec_peaks[["mz", "intensity"]] = pd.DataFrame(
    spec_peaks["peaks"].tolist(), index=spec_peaks.index
)
spec_peaks = spec_peaks.drop(columns=["peaks"])
spec_peaks["mz"] = spec_peaks["mz"].astype(float)
spec_peaks["intensity"] = spec_peaks["intensity"].astype(float)

peaks_with_ints = peaks.merge(
    spec_peaks.rename(columns={"mz": "peak_mz"}),
    on=["spec_id", "peak_mz"],
    how="inner",
)

total_int = (
    peaks_with_ints.groupby("spec_id")["intensity"].sum().rename("total")
)
iso_int = (
    peaks_with_ints[peaks_with_ints["isotope"] > 0]
    .groupby("spec_id")["intensity"].sum()
    .rename("iso")
)
int_frac = pd.concat([total_int, iso_int], axis=1).fillna(0)
int_frac["iso_frac"] = int_frac["iso"] / int_frac["total"]

print("Per-spectrum isotope intensity fraction:")
print(int_frac["iso_frac"].describe())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(int_frac["iso_frac"], bins=50, color="mediumseagreen", edgecolor="white")
ax.set_xlabel("Fraction of annotated intensity from isotope peaks")
ax.set_ylabel("Number of spectra")
ax.set_title("Isotope intensity fraction per spectrum (NIST20)")
plt.tight_layout()
plt.show()